# Level 4: Constraint Closure, Cohomological SES, and DDA Cascade Engine

**Paper Package 002** — Pedagogical Progression: Level 4 of 5 (Formerly Reviewer Notebook)

**Target Audience:** Postdoctoral researchers and advanced theorists in quantum gravity, topological field theory, and discrete geometry.

**Core Idea:** The quotient space $Q^1(K) = C^1(K) / \operatorname{im} d_0$ sits inside a fundamental short exact sequence:
$$0 \longrightarrow H^1(K) \longrightarrow Q^1(K) \xrightarrow{\;\bar d_1\;} \operatorname{im} d_1 \longrightarrow 0.$$
This separates globally inexact closed cochains (harmonic classes in $H^1$) from failures of face closure (boundary fluxes in $\operatorname{im} d_1$). Adding higher simplices alters $d_1$ without reducing $Q^1$, transforming holonomies into face defects.

*Evidence scope: finite toy-model demonstration only.*

In [ ]:
import sys
from pathlib import Path
import numpy as np

# Robust sys.path resolution for code/ and omega_engine
cwd = Path.cwd().resolve()
for candidate in (cwd, cwd / "code", cwd.parent / "code", *(d / "code" for d in cwd.parents)):
    if (candidate / "constraint_closure_l1_structure.py").exists():
        if str(candidate) not in sys.path:
            sys.path.insert(0, str(candidate))
        for p in candidate.parents:
            if (p / "omega_engine").exists() and str(p) not in sys.path:
                sys.path.insert(0, str(p))
                break
        break

from constraint_closure_l1_structure import ConstraintClosureL1Structure
from l1_topological_defects import (
    evaluate_c3_hodge_split_inflation,
    derive_fermion_mass_spectrum_lggr,
    derive_cosmological_constant_residual,
)
from dda_trace_recorder import DDATraceRecorder
from solver_of_solvers import CascadeSolver, Network
print('[OK] Level 4 solvers, DDA engine, and trace recorder loaded successfully!')

## 1. 3-Cycle $C_3$ Worst-Case Hodge Split Inflation

We evaluate the worst-case inflation ratio between the Euclidean Hodge harmonic split and the true $\ell^1$ quotient norm on $C_3$, achieving the sharp ceiling $R_{\mathrm{split}}^{\max}(C_3) = 3 - 2/3 = 7/3$:

In [ ]:
res_c3 = evaluate_c3_hodge_split_inflation(cochain_omega=[0.0, 1.0, 0.0])
print('=== Demo 1: 3-Cycle C3 Hodge Split Inflation ===')
print(f'Cochain omega:              {res_c3["cochain_omega"]}')
print(f'L2 Exact d0_alpha:         {res_c3["d0_alpha_l2"]}')
print(f'L2 Harmonic h:             {res_c3["h_l2_harmonic"]}')
print(f'Bookkeeping R_split:       {res_c3["r_split"]:.6f} (Analytical Ceiling: {res_c3["r_split_analytic_ceiling"]:.6f})')
print(f'Representative R_rep:      {res_c3["r_rep"]:.6f}')
print(f'L1 Quotient Norm Phi1:     {res_c3["phi_1"]:.6f}')
print(f'Is Sharp Ceiling:          {res_c3["is_sharp_c3_ceiling"]}')

assert np.isclose(res_c3["r_split"], 7.0 / 3.0)
assert np.isclose(res_c3["phi_1"], 1.0)
assert res_c3["is_sharp_c3_ceiling"] is True
print('[OK] Demo 1 verified mathematically!')

## 2. LGGR Fermion Generation Mass Spectrum

We evaluate the $\mathbb{Z}_3$ cyclic Laplacian eigenvalue ratio on a 3-cycle, anchored to the Lean 4 proof witness `localGreenGlobalRed_pair_obstruction`:

In [ ]:
res_fermion = derive_fermion_mass_spectrum_lggr(cochain_1cycle=[1.0, 1.0, 1.0], v_ew_gev=246.0)
print('=== Demo 2: LGGR Fermion Generation Mass Spectrum ===')
print(f'Lean 4 Proof Witness:      {res_fermion["lean4_proof_witness"]}')
print(f'L1 Primitive Difference:   {res_fermion["phi_l1_primitive_difference"]:.6f}')
print(f'Face Stiffness Multiplier: {res_fermion["face_stiffness_multiplier"]:.6f}')
print(f'Z3 Spectral Eigenvalues:  {[round(x, 4) for x in res_fermion["z3_spectral_eigenvalues"]]}')
print(f'Raw Mass Gaps:             {[round(x, 4) for x in res_fermion["raw_residual_mass_gaps"]]}')

# Verify exact relative eigenvalue ratio 0 : sqrt(3) : sqrt(3)
eigs = res_fermion["z3_spectral_eigenvalues"]
assert np.isclose(eigs[0], 0.0)
assert np.isclose(eigs[1], np.sqrt(3.0))
assert np.isclose(eigs[2], np.sqrt(3.0))
print('[OK] Demo 2 eigenvalue spectrum verified!')

## 3. 4D Regge Cosmological Constant Residual Toy

We evaluate the toy cosmological constant proxy computed as the un-repaired global $\ell^1$ deficit density of a 4D Regge simplicial block:

In [ ]:
res_lambda = derive_cosmological_constant_residual(hinge_valencies_4d=[5, 4, 5, 4], immirzi_gamma=0.274066)
print('=== Demo 3: Cosmological Constant Residual Density ===')
print(f'Lean 4 Proof Witness:      {res_lambda["lean4_proof_witness"]}')
print(f'4D Deficit Angles (rad):   {[round(x, 4) for x in res_lambda["epsilon_4d_rad"]]}')
print(f'Total L1 Residual:         {res_lambda["total_l1_residual"]:.6f}')
print(f'Cell 4-Volume (L_P^4):      {res_lambda["total_cell_volume_l_p4"]:.1f}')
print(f'Lambda Residual Density:    {res_lambda["lambda_residual_density"]:.6f}')

assert res_lambda["total_l1_residual"] > 0.0
assert res_lambda["lambda_residual_density"] > 0.0
print('[OK] Demo 3 deficit calculations verified!')

## 4. Complete Measured Run: Real Execution & DDA Receipt

This cell executes the **complete measured run** milestone:
1. An input cochain $\omega$ on an admitted simplicial complex.
2. Actual quotient optimization $\Phi_1([\omega])$ via Linear Programming.
3. Stepwise DDA execution recorded in `DDATraceRecorder`.
4. Recomputed residuals at every stage.
5. Verification of the `SolverCascadeReceipt` with degree and prior obligations preserved.

In [ ]:
# Construct a 2-triangle network sharing an edge
net = Network(
    nodes=[0, 1, 2, 3],
    edges=[(0, 1), (1, 2), (2, 0), (1, 3), (3, 0)],
    faces=[(0, 1, 2), (0, 1, 3)],
)
solver = CascadeSolver(network=net)
recorder = DDATraceRecorder()

# Input cochain with exact gauge noise on edge (0, 1) and circulating cycle flux
omega = {(0, 1): 2.5, (1, 2): 1.0, (2, 0): 1.0, (1, 3): 0.5, (3, 0): 0.5}

# 1. Foundation step
found = solver.dda_engine.run_foundation(omega)
rec_f = recorder.record_step('FOUNDATION', 'E', found.phi1, found.phi1, {'phi1': found.phi1})

# 2. Step DDA gauge repair (exact d0 projection)
cert_g, (phi1_g, gamma2_g, chi_g) = solver.dda_engine.step_dda(move_type='exact_d0')
rec_g = recorder.record_step('DDA_STEP', 'R', found.phi1, phi1_g, {'phi1': phi1_g})

# 3. Step DDA multi-cycle descent
cert_m, (phi1_m, gamma2_m, chi_m) = solver.dda_engine.step_dda(move_type='multi_cycle_l1')
rec_m = recorder.record_step('DDA_STEP', 'C', phi1_g, phi1_m, {'phi1': phi1_m})

# 4. Final cascade execution and receipt generation
receipt = solver.solve(edge_values=omega, max_lifts=1)

print('=== Complete Measured Run Execution ===')
print(f'Initial residual Phi_1:     {receipt.initial_residual_size:.6f}')
print(f'Terminal status:            {receipt.terminal_status}')
print(f'Final residual size:        {receipt.final_residual_size:.6f}')
print(f'Degree reached:             {receipt.degree_window_reached}')
print(f'Final residual triple:      {receipt.final_residual_triple}')
print(f'Prior degree obligations:   {receipt.prior_degree_obligations}')
print(f'Real DDA tape tokens:       {" -> ".join(recorder.tape)}')

assert receipt.initial_residual_size >= 0.0
assert len(recorder.records) >= 3
assert receipt.authority_effect == 'None'
print('[OK] Complete measured run executed and verified cleanly!')